In [86]:
# import modules
import os
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from natsort import natsorted

In [87]:
# set the base directory for the project
cwd = os.getcwd()
BASE_DIR = os.path.abspath(os.path.join(cwd, "..", ".."))

# build paths inside the repo
get_data_path = lambda folders, fname: os.path.normpath(
    os.path.join(BASE_DIR, *folders, fname)
)

crispr_screens_path = get_data_path(['data', 'output', 'processed_CRISPR_screens'], '')

ito_GImap_file_path = get_data_path(['data', 'input', 'GIMAP'], 'ito_gimap_sl_labels.tsv')
klingbeil_GImap_file_path = get_data_path(['data', 'input', 'GIMAP'], 'klingbeil_gimap_sl_labels.tsv')
harle_GImap_file_path = get_data_path(['data', 'input', 'GIMAP'], 'harle_gimap_common10_lfc_labels_updated.tsv')

file_path_sample_info = get_data_path(['data', 'input', 'DepMap22Q4'], 'sample_info.csv')

file_path_genenames = get_data_path(['data', 'input', 'other'], 'approved_and_previous_symbols.csv')

#### Processing GI and LFC score of Ito

In [88]:
ito_gi_score = pd.read_csv(ito_GImap_file_path, sep='\t')
ito_gi_score.columns

Index(['cell_line', 'DepMap_ID', 'gene_pair', 'gi_score', 'mean_observed_lfc',
       'mean_expected_lfc', 'n_gi_observations', 'p_val', 'fdr',
       'gi_score_threshold', 'lfc_threshold', 'passes_gi_threshold',
       'passes_lfc_threshold', 'sl_label'],
      dtype='object')

In [89]:
ito_gi_score = ito_gi_score.rename(columns={'gene_pair': 'genepair', 'sl_label': 'SL_new'})
ito_gi_score['SL_new'] = ito_gi_score['SL_new'].astype(bool)
ito_gi_score = ito_gi_score[['genepair', 'DepMap_ID', 'cell_line', 'SL_new']].copy()

In [90]:
# check the gene pairs 
# is ADSS1_ADSS2 or ADSS_ADSSL1 in the GIMAP data?

display(ito_gi_score.loc[ito_gi_score['genepair'] == 'ADSS_ADSSL1',])
display(ito_gi_score.loc[ito_gi_score['genepair'] == 'ADSS1_ADSS2',])

,genepair,DepMap_ID,cell_line,SL_new
228,ADSS_ADSSL1,ACH-000681,A549,False
5285,ADSS_ADSSL1,ACH-000756,GI1_004,False
10342,ADSS_ADSSL1,ACH-000801,HS936T,False
15399,ADSS_ADSSL1,ACH-000632,HS944T,False
20456,ADSS_ADSSL1,ACH-001524,HSC5,False
25513,ADSS_ADSSL1,ACH-000915,IPC298,False
30570,ADSS_ADSSL1,ACH-001554,MEL202_003,False
35627,ADSS_ADSSL1,ACH-000987,MEWO,False
40684,ADSS_ADSSL1,ACH-000881,Meljuso,False
45741,ADSS_ADSSL1,ACH-000022,PATU8988S,False


,genepair,DepMap_ID,cell_line,SL_new


#### Processing GI and LFC score of Klingbeil

In [91]:
kln_gi_score = pd.read_csv(klingbeil_GImap_file_path, sep='\t')
kln_gi_score.columns

Index(['cell_line', 'DepMap_ID', 'gene_pair', 'gi_score', 'mean_observed_lfc',
       'mean_expected_lfc', 'n_gi_observations', 'p_val', 'fdr',
       'gi_score_threshold', 'lfc_threshold', 'passes_gi_threshold',
       'passes_lfc_threshold', 'sl_label'],
      dtype='object')

In [118]:
kln_gi_score.loc[kln_gi_score['genepair'] == 'AKT1_AKT2',]

,genepair,A1,A2,DepMap_ID,cell_line,SL_new
139,AKT1_AKT2,AKT1,AKT2,ACH-000681,A549,False
2668,AKT1_AKT2,AKT1,AKT2,ACH-000222,ASPC1,True
5197,AKT1_AKT2,AKT1,AKT2,ACH-000187,CORL311,False
7717,AKT1_AKT2,AKT1,AKT2,NaN,CTR,True
10246,AKT1_AKT2,AKT1,AKT2,ACH-000866,H1048,True
12775,AKT1_AKT2,AKT1,AKT2,ACH-000510,H1299,False
15304,AKT1_AKT2,AKT1,AKT2,ACH-000830,H1436,False
17833,AKT1_AKT2,AKT1,AKT2,ACH-000559,H1836,False
20362,AKT1_AKT2,AKT1,AKT2,ACH-000290,H209,False
22891,AKT1_AKT2,AKT1,AKT2,ACH-000639,H211,False


In [92]:
kln_gi_score = kln_gi_score.rename(columns={'gene_pair': 'genepair', 'sl_label': 'SL_new'})
kln_gi_score['SL_new'] = kln_gi_score['SL_new'].astype(bool)
kln_gi_score = kln_gi_score[['genepair', 'DepMap_ID', 'cell_line', 'SL_new']].copy()

In [93]:
# check the gene pairs 
# is ADSS1_ADSS2 or ADSS_ADSSL1 in the GIMAP data?

display(kln_gi_score.loc[kln_gi_score['genepair'] == 'ARHGAP15_ARHGAP9',])
display(kln_gi_score.loc[kln_gi_score['genepair'] == 'ARHGAP9_ARHGAP15',])

,genepair,DepMap_ID,cell_line,SL_new
221,ARHGAP15_ARHGAP9,ACH-000681,A549,False
2750,ARHGAP15_ARHGAP9,ACH-000222,ASPC1,False
5279,ARHGAP15_ARHGAP9,ACH-000187,CORL311,False
7799,ARHGAP15_ARHGAP9,NaN,CTR,False
10328,ARHGAP15_ARHGAP9,ACH-000866,H1048,False
12857,ARHGAP15_ARHGAP9,ACH-000510,H1299,False
15386,ARHGAP15_ARHGAP9,ACH-000830,H1436,False
17915,ARHGAP15_ARHGAP9,ACH-000559,H1836,False
20444,ARHGAP15_ARHGAP9,ACH-000290,H209,False
22973,ARHGAP15_ARHGAP9,ACH-000639,H211,False


,genepair,DepMap_ID,cell_line,SL_new


#### Processing GI and LFC score of Harle

In [94]:
harle_gi_score = pd.read_csv(harle_GImap_file_path, sep='\t')
harle_gi_score.columns

Index(['cell_line', 'gene_pair', 'raw_gene_pair', 'score', 'mean_observed_lfc',
       'mean_expected_lfc', 'n_gi_observations', 'p_val', 'fdr',
       'gemini_strong_score', 'gemini_sensitive_lethality',
       'gemini_sensitive_recovery', 'score_threshold', 'lfc_threshold',
       'passes_score_threshold', 'passes_lfc_threshold', 'sl_label'],
      dtype='object')

In [95]:
sample_info = pd.read_csv(file_path_sample_info)
CCLE_name_to_DepMapID = dict(zip(sample_info.cell_line_name, sample_info.DepMap_ID))
CCLE_name_to_DepMapID['KP-1N'] = 'ACH-001107'

In [96]:
harle_gi_score = harle_gi_score.rename(columns={'sl_label': 'SL_new'})
harle_gi_score['SL_new'] = harle_gi_score['SL_new'].astype(bool)
harle_gi_score['DepMap_ID'] = harle_gi_score['cell_line'].map(CCLE_name_to_DepMapID)
harle_gi_score = harle_gi_score[['gene_pair', 'raw_gene_pair', 'DepMap_ID', 'cell_line', 'SL_new']].copy()

In [97]:
# check the gene pairs 
# is ADSS1_ADSS2 or ADSS_ADSSL1 in the GIMAP data?

display(harle_gi_score.loc[harle_gi_score['raw_gene_pair'] == 'ADSS_ADSSL1',])
display(harle_gi_score.loc[harle_gi_score['raw_gene_pair'] == 'ADSS1_ADSS2',])

,gene_pair,raw_gene_pair,DepMap_ID,cell_line,SL_new
10,ADSS;ADSSL1,ADSS_ADSSL1,ACH-000219,A-375,False
482,ADSS;ADSSL1,ADSS_ADSSL1,ACH-000788,A2058,False
954,ADSS;ADSSL1,ADSS_ADSSL1,ACH-000681,A549,False
1426,ADSS;ADSSL1,ADSS_ADSSL1,ACH-000222,AsPC-1,False
1898,ADSS;ADSSL1,ADSS_ADSSL1,ACH-000535,BxPC-3,False
2370,ADSS;ADSSL1,ADSS_ADSSL1,NaN,C092,False
2842,ADSS;ADSSL1,ADSS_ADSSL1,ACH-000138,CFPAC-1,False
3314,ADSS;ADSSL1,ADSS_ADSSL1,ACH-001024,CHL-1,True
3786,ADSS;ADSSL1,ADSS_ADSSL1,ACH-000662,COR-L23,False
4258,ADSS;ADSSL1,ADSS_ADSSL1,ACH-000354,Capan-1,False


,gene_pair,raw_gene_pair,DepMap_ID,cell_line,SL_new


### Add gene names 

In [98]:
# read the gene names mapping file
id_map = pd.read_csv(file_path_genenames)

# create dictionaries to map gene symbols to Entrez IDs
approved_sym_to_entrez_id = dict(zip(id_map['Approved symbol'], id_map['entrez_id']))
entrezid_to_symbol = dict(zip(id_map['entrez_id'], id_map['Approved symbol']))

# create dictionaries to map previous gene symbols to Entrez IDs
id_map_cleaned = id_map.dropna(axis=0, how='any', subset=['Previous symbol', 'entrez_id']).reset_index(drop=True)
prev_sym_to_entrez_id = dict(zip(id_map_cleaned['Previous symbol'], id_map_cleaned['entrez_id']))

In [99]:
def process_gene_symbols(df, id_map, genepair_col):
    # Split gene pairs into two columns
    df.insert(1, "A1", df[genepair_col].apply(lambda x: x.split("_", 1)[0]))
    df.insert(2, "A2", df[genepair_col].apply(lambda x: x.split("_", 1)[1]))

    # Assign mapped NCBI Gene IDs to A1 and A2
    df = df.assign(
        A1_entrez = df['A1'].map(approved_sym_to_entrez_id),
        A2_entrez = df['A2'].map(approved_sym_to_entrez_id))

    df['A1_entrez'] = df['A1_entrez'].fillna(df['A1'].map(prev_sym_to_entrez_id))
    df['A2_entrez'] = df['A2_entrez'].fillna(df['A2'].map(prev_sym_to_entrez_id))

    # Drop rows with unresolved NCBI Gene IDs
    df = df.dropna(subset=['A1_entrez', 'A2_entrez'], how='any').reset_index(drop=True)
    df = df.drop(genepair_col, axis=1)

    df.rename(columns={'A1': 'org_A1', 'A2': 'org_A2'}, inplace=True)

    df.insert(1, 'A1', df['A1_entrez'].map(entrezid_to_symbol))
    df.insert(2, 'A2', df['A2_entrez'].map(entrezid_to_symbol))

    list_c = [[x, y] for x, y in zip(df.A1, df.A2)]
    genepairs = ['_'.join(natsorted(pair)) for pair in list_c]
    df.insert(0, 'genepair', genepairs, True)

    df = df[['genepair', 'A1', 'A2', 'A1_entrez', 'A2_entrez', 'DepMap_ID', 'SL_new', 'org_A1', 'org_A2']]
    return df

In [100]:
ito_df_melt = process_gene_symbols(ito_gi_score, id_map, genepair_col='genepair')
kln_df_melt = process_gene_symbols(kln_gi_score, id_map, genepair_col='genepair')
harle_df_melt = process_gene_symbols(harle_gi_score, id_map, genepair_col='raw_gene_pair')

In [101]:
# Function to sort each pair of gene symbols and their Entrez IDs
def sort_gene_pairs(row):
    # Sort the genes alphabetically and determine new order
    sorted_genes = natsorted([row['A1'], row['A2']])
    
    # Match the sorted genes to the original ones and rearrange Entrez IDs accordingly
    if sorted_genes[0] == row['A1']:
        return pd.Series([sorted_genes[0], sorted_genes[1], row['A1_entrez'], row['A2_entrez']])
    else:
        return pd.Series([sorted_genes[0], sorted_genes[1], row['A2_entrez'], row['A1_entrez']])

In [102]:
# Define the function to process the DataFrame
def process_dataframe(df):
    # Apply the sorting to each row
    df[['A1_sorted', 'A2_sorted', 'A1_entrez_sorted', 'A2_entrez_sorted']] = df.apply(sort_gene_pairs, axis=1)
    
    # Drop the old columns and rename the new ones
    df = df.drop(columns=['A1', 'A2', 'A1_entrez', 'A2_entrez']).copy()
    df = df.rename(columns={
        'A1_sorted': 'A1',
        'A2_sorted': 'A2',
        'A1_entrez_sorted': 'A1_entrez',
        'A2_entrez_sorted': 'A2_entrez'
    })
    
    return df

In [103]:
gimap_ito_df = process_dataframe(ito_df_melt)
gimap_kln_df = process_dataframe(kln_df_melt)
gimap_harle_df = process_dataframe(harle_df_melt)

In [104]:
display(gimap_ito_df.loc[gimap_ito_df['genepair'] == 'ADSS1_ADSS2'])
display(gimap_kln_df.loc[gimap_kln_df['genepair'] == 'ARHGAP9_ARHGAP15'])
display(gimap_harle_df.loc[gimap_harle_df['genepair'] == 'ADSS1_ADSS2'])

,genepair,DepMap_ID,SL_new,org_A1,org_A2,A1,A2,A1_entrez,A2_entrez
228,ADSS1_ADSS2,ACH-000681,False,ADSS,ADSSL1,ADSS1,ADSS2,122622.0,159.0
5283,ADSS1_ADSS2,ACH-000756,False,ADSS,ADSSL1,ADSS1,ADSS2,122622.0,159.0
10338,ADSS1_ADSS2,ACH-000801,False,ADSS,ADSSL1,ADSS1,ADSS2,122622.0,159.0
15393,ADSS1_ADSS2,ACH-000632,False,ADSS,ADSSL1,ADSS1,ADSS2,122622.0,159.0
20448,ADSS1_ADSS2,ACH-001524,False,ADSS,ADSSL1,ADSS1,ADSS2,122622.0,159.0
25503,ADSS1_ADSS2,ACH-000915,False,ADSS,ADSSL1,ADSS1,ADSS2,122622.0,159.0
30558,ADSS1_ADSS2,ACH-001554,False,ADSS,ADSSL1,ADSS1,ADSS2,122622.0,159.0
35613,ADSS1_ADSS2,ACH-000987,False,ADSS,ADSSL1,ADSS1,ADSS2,122622.0,159.0
40668,ADSS1_ADSS2,ACH-000881,False,ADSS,ADSSL1,ADSS1,ADSS2,122622.0,159.0
45723,ADSS1_ADSS2,ACH-000022,False,ADSS,ADSSL1,ADSS1,ADSS2,122622.0,159.0


,genepair,DepMap_ID,SL_new,org_A1,org_A2,A1,A2,A1_entrez,A2_entrez
212,ARHGAP9_ARHGAP15,ACH-000681,False,ARHGAP15,ARHGAP9,ARHGAP9,ARHGAP15,64333.0,55843.0
2713,ARHGAP9_ARHGAP15,ACH-000222,False,ARHGAP15,ARHGAP9,ARHGAP9,ARHGAP15,64333.0,55843.0
5214,ARHGAP9_ARHGAP15,ACH-000187,False,ARHGAP15,ARHGAP9,ARHGAP9,ARHGAP15,64333.0,55843.0
7706,ARHGAP9_ARHGAP15,NaN,False,ARHGAP15,ARHGAP9,ARHGAP9,ARHGAP15,64333.0,55843.0
10207,ARHGAP9_ARHGAP15,ACH-000866,False,ARHGAP15,ARHGAP9,ARHGAP9,ARHGAP15,64333.0,55843.0
12708,ARHGAP9_ARHGAP15,ACH-000510,False,ARHGAP15,ARHGAP9,ARHGAP9,ARHGAP15,64333.0,55843.0
15209,ARHGAP9_ARHGAP15,ACH-000830,False,ARHGAP15,ARHGAP9,ARHGAP9,ARHGAP15,64333.0,55843.0
17710,ARHGAP9_ARHGAP15,ACH-000559,False,ARHGAP15,ARHGAP9,ARHGAP9,ARHGAP15,64333.0,55843.0
20211,ARHGAP9_ARHGAP15,ACH-000290,False,ARHGAP15,ARHGAP9,ARHGAP9,ARHGAP15,64333.0,55843.0
22712,ARHGAP9_ARHGAP15,ACH-000639,False,ARHGAP15,ARHGAP9,ARHGAP9,ARHGAP15,64333.0,55843.0


,genepair,DepMap_ID,SL_new,org_A1,org_A2,A1,A2,A1_entrez,A2_entrez
10,ADSS1_ADSS2,ACH-000219,False,ADSS,ADSSL1,ADSS1,ADSS2,122622.0,159.0
482,ADSS1_ADSS2,ACH-000788,False,ADSS,ADSSL1,ADSS1,ADSS2,122622.0,159.0
954,ADSS1_ADSS2,ACH-000681,False,ADSS,ADSSL1,ADSS1,ADSS2,122622.0,159.0
1426,ADSS1_ADSS2,ACH-000222,False,ADSS,ADSSL1,ADSS1,ADSS2,122622.0,159.0
1898,ADSS1_ADSS2,ACH-000535,False,ADSS,ADSSL1,ADSS1,ADSS2,122622.0,159.0
2370,ADSS1_ADSS2,NaN,False,ADSS,ADSSL1,ADSS1,ADSS2,122622.0,159.0
2842,ADSS1_ADSS2,ACH-000138,False,ADSS,ADSSL1,ADSS1,ADSS2,122622.0,159.0
3314,ADSS1_ADSS2,ACH-001024,True,ADSS,ADSSL1,ADSS1,ADSS2,122622.0,159.0
3786,ADSS1_ADSS2,ACH-000662,False,ADSS,ADSSL1,ADSS1,ADSS2,122622.0,159.0
4258,ADSS1_ADSS2,ACH-000354,False,ADSS,ADSSL1,ADSS1,ADSS2,122622.0,159.0


In [105]:
crispr_files = [
    '/Users/narod/Library/CloudStorage/GoogleDrive-narod.kebabci@ucdconnect.ie/My Drive/GitRepos/context_specific_SL_prediction/data/output/processed_CRISPR_screens/processed_ito_df_scored.csv',
    '/Users/narod/Library/CloudStorage/GoogleDrive-narod.kebabci@ucdconnect.ie/My Drive/GitRepos/context_specific_SL_prediction/data/output/processed_CRISPR_screens/processed_klingbeil_df_scored.csv',
    '/Users/narod/Library/CloudStorage/GoogleDrive-narod.kebabci@ucdconnect.ie/My Drive/GitRepos/context_specific_SL_prediction/data/output/processed_CRISPR_screens/processed_harle_df_scored.csv'
]

In [106]:
ito_annotated = pd.read_csv(crispr_files[0])
ito_annotated = pd.merge(ito_annotated, gimap_ito_df.drop(['A1', 'A2', 'org_A2', 'org_A1', 'A1_entrez', 'A2_entrez'], axis=1), 
                         on=['genepair', 'DepMap_ID'], how='left')
ito_annotated[:3]

,genepair,A1,A2,DepMap_ID,cell_line,Gemini_FDR,raw_LFC,SL,org_A1,org_A2,...,either_in_complex,mean_complex_essentiality,colocalisation,interact,n_total_ppi,fet_ppi_overlap,gtex_spearman_corr,gtex_min_mean_expr,gtex_max_mean_expr,SL_new
0,A3GALT2_ABO,A3GALT2,ABO,ACH-000022,PATU8988S_PANCREAS,0.998944,0.088856,False,A3GALT2,ABO,...,0.0,0.0,0.0,0.0,3.0,0.0,0.114847,0.258739,11.702,False
1,A3GALT2_ABO,A3GALT2,ABO,ACH-000307,PK1_PANCREAS,0.986587,0.201704,False,A3GALT2,ABO,...,0.0,0.0,0.0,0.0,3.0,0.0,0.114847,0.258739,11.702,False
2,A3GALT2_ABO,A3GALT2,ABO,ACH-000632,HS944T_SKIN,1.000000,0.069772,False,A3GALT2,ABO,...,0.0,0.0,0.0,0.0,3.0,0.0,0.114847,0.258739,11.702,False


In [107]:
ito_annotated.loc[ito_annotated['SL_new'].isna(), 'genepair'].unique()

array(['CBS_CBSL', 'CELA1_CTRL', 'CELA2A_CTRL', 'CELA2B_CTRL',
       'CELA3A_CTRL', 'CELA3B_CTRL', 'CTRB1_CTRL', 'CTRB2_CTRL',
       'CTRC_CTRL'], dtype=object)

In [108]:
raw_kln_annotated = pd.read_csv(crispr_files[1])

In [109]:
raw_kln_annotated = pd.read_csv(crispr_files[1])
kln_annotated = pd.merge(raw_kln_annotated, gimap_kln_df.drop(['A1', 'A2', 'org_A2', 'org_A1', 'A1_entrez', 'A2_entrez'], axis=1), 
                         on=['genepair','DepMap_ID'], how='left')
kln_annotated[:3]

,GENE_COMBINATION,domain_combination,genepair,A1,A2,cell_line,DepMap_ID,GEMINI,LFC,FDR,...,either_in_complex,mean_complex_essentiality,colocalisation,interact,n_total_ppi,fet_ppi_overlap,gtex_spearman_corr,gtex_min_mean_expr,gtex_max_mean_expr,SL_new
0,AAK1:Kinase_domain;BMP2K:Kinase_domain,Kinase_domain_Kinase_domain,AAK1_BMP2K,AAK1,BMP2K,HEL,ACH-000004,0.218665,0.092748,0.559622,...,0.0,0.0,0.0,0.0,77.0,21.867726,0.261701,6.713555,6.761786,False
1,AAK1:Kinase_domain;BMP2K:Kinase_domain,Kinase_domain_Kinase_domain,AAK1_BMP2K,AAK1,BMP2K,T3M4,ACH-000085,0.205641,0.271945,0.844509,...,0.0,0.0,0.0,0.0,77.0,21.867726,0.261701,6.713555,6.761786,False
2,AAK1:Kinase_domain;BMP2K:Kinase_domain,Kinase_domain_Kinase_domain,AAK1_BMP2K,AAK1,BMP2K,HPAFII,ACH-000094,0.044486,0.596616,0.940129,...,0.0,0.0,0.0,0.0,77.0,21.867726,0.261701,6.713555,6.761786,False


In [110]:
kln_annotated.loc[kln_annotated['SL_new'].isna(), 'genepair'].unique()

array(['AAK1_BMP2K', 'AATK_LMTK2', 'AATK_LMTK3', ..., 'ZFYVE21_ZFYVE28',
       'ZFYVE9_ZFYVE16', 'ZMYND8_ZMYND11'], dtype=object)

In [111]:
harle_annotated = pd.read_csv(crispr_files[2])
harle_annotated = pd.merge(harle_annotated, gimap_harle_df.drop(['A1', 'A2', 'org_A2', 'org_A1', 'A1_entrez', 'A2_entrez'], axis=1), 
                         on=['genepair', 'DepMap_ID'], how='left')
harle_annotated[:3]

,genepair,sorted_gene_pair,A1,A2,cell_line,DepMap_ID,mean_norm_gi,fdr,is_bassik_hit,targetA__is_single_depleted,...,either_in_complex,mean_complex_essentiality,colocalisation,interact,n_total_ppi,fet_ppi_overlap,gtex_spearman_corr,gtex_min_mean_expr,gtex_max_mean_expr,SL_new
0,ABL1_ABL2,ABL1|ABL2,ABL1,ABL2,HPAF-II,ACH-000094,0.461745,0.992489,0,0,...,1.0,0.238134,0.5,1.0,208.0,40.51941,0.69316,7.660549,61.157812,False
1,ABL1_ABL2,ABL1|ABL2,ABL1,ABL2,SU.86.86,ACH-000114,0.281467,0.940366,0,0,...,1.0,0.238134,0.5,1.0,208.0,40.51941,0.69316,7.660549,61.157812,False
2,ABL1_ABL2,ABL1|ABL2,ABL1,ABL2,CFPAC-1,ACH-000138,0.273686,0.839309,0,0,...,1.0,0.238134,0.5,1.0,208.0,40.51941,0.69316,7.660549,61.157812,False


In [112]:
harle_annotated.loc[harle_annotated['SL_new'].isna(), 'genepair'].unique()

array([], dtype=object)

In [113]:
ito_annotated = ito_annotated.loc[~ito_annotated['SL_new'].isna()].reset_index(drop=True)

print('ito')
display(ito_annotated['SL_new'].value_counts())
print('')

ito


SL_new
False    49215
True       450
Name: count, dtype: int64

In [114]:
kln_annotated = kln_annotated.loc[~kln_annotated['SL_new'].isna()].reset_index(drop=True)

print('kln')
print(f'Number of gene pairs: {kln_annotated.genepair.nunique()}')
print(f'Number of cell lines: {kln_annotated.DepMap_ID.nunique()}')
display(kln_annotated['SL_new'].value_counts())
print('')

kln
Number of gene pairs: 2239
Number of cell lines: 21


SL_new
False    49830
True      1011
Name: count, dtype: int64

In [115]:
harle_annotated = harle_annotated.loc[~harle_annotated['SL_new'].isna()].reset_index(drop=True)

print('harle')
display(harle_annotated['SL_new'].value_counts())
print('')

harle


SL_new
False    8335
True      656
Name: count, dtype: int64

In [85]:
annotated_datasets = [ito_annotated, kln_annotated, harle_annotated]

In [ ]:
output_dir = get_data_path(['data', 'output', 'processed_CRISPR_screens'], '')

filenames = ['processed_ito_df', 'processed_klingbeil_df', 'processed_harle_df']

for i, df in enumerate(annotated_datasets):
    base_filename = filenames[i]
    output_path = os.path.join(output_dir, f"{base_filename}_GImap_labelled.csv")
    df.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")